In [64]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import plotly.express as px
from helper_functions import quick_px_scattermap

In [65]:
station_avg_delay = pd.read_csv("../data/processed/cleaned_delay_data.csv")
station_avg_ridership = pd.read_csv("../data/processed/selected_station_ridership.csv")
sj_adt = pd.read_csv("../data/processed/selected_sj_adt.csv")

selected_routes = ['22', 'Rapid 522', '23', 'Rapid 523', '25', '60']
station_avg_delay = station_avg_delay[station_avg_delay["route_id"].isin(selected_routes)]
station_avg_delay = station_avg_delay.groupby(['route_id', 'direction_id', 'stop_id', 'stop_sequence']).agg(mean_delay=("computed_delay_sec", "mean")).reset_index().sort_values(['route_id', 'direction_id', 'stop_sequence'])
station_avg_delay["route_id"] = station_avg_delay["route_id"].str.replace(
    r"^Rapid\s+", "", regex=True
).astype(int)
station_avg_delay

,route_id,direction_id,stop_id,stop_sequence,mean_delay
18,22,0,60328,2,139.500000
19,22,0,60329,3,196.000000
20,22,0,60330,4,173.000000
21,22,0,60331,5,146.500000
22,22,0,60332,6,158.666667
...,...,...,...,...,...
586,523,1,60651,13,-466.000000
587,523,1,60653,14,-488.500000
588,523,1,60656,15,-525.000000
589,523,1,60660,16,-604.500000


In [66]:
station_avg_ridership['direction_id'] = station_avg_ridership['direction_id'] % 10
station_avg_ridership = station_avg_ridership.sort_values(['route_id', 'direction_id', 'stop_id'])
station_avg_ridership

,route_id,direction_id,stop_id,stop_name,boardings,alightings,total_b_a,geometry
0,22,0,60001,Santa Clara Transit Center,152.880022,110.852250,263.732272,POINT (6144099.62758 1954225.5200159)
4,22,0,60020,El Camino & Lafayette,40.890986,57.438087,98.329073,POINT (6141651.00288408 1954992.24617273)
6,22,0,60035,King & Alum Rock,149.607974,157.936236,307.544210,POINT (6167727.93202934 1953654.17929506)
7,22,0,60036,King & San Antonio,36.418456,38.469837,74.888293,POINT (6168552.89461824 1952565.84026439)
8,22,0,60037,King & Hermocilla,31.495157,45.198273,76.693430,POINT (6169311.51383591 1951509.07203673)
...,...,...,...,...,...,...,...,...
586,523,1,64478,Lockheed Martin Transit Center,0.000000,317.466630,317.466630,POINT (6117801.90097608 1975568.38743506)
601,523,1,64696,Stevens Creek & Winchester,85.577443,90.755989,176.333432,POINT (6139677.16759291 1943518.2914304)
620,523,1,65741,Stevens Creek & Valley Fair / Santana Row,109.060532,391.806776,500.867308,POINT (6141126.594156 1943538.09191573)
642,523,1,65902,Sunnyvale-Saratoga & El Camino,189.107898,219.736022,408.843920,POINT (6116423.22131875 1959757.18710214)


In [67]:
stations_df = pd.merge(station_avg_delay, station_avg_ridership, on=['route_id', 'direction_id', 'stop_id'], how='left')
stations_df

,route_id,direction_id,stop_id,stop_sequence,mean_delay,stop_name,boardings,alightings,total_b_a,geometry
0,22,0,60328,2,139.500000,Palo Alto Transit Center (Bay 10),733.994859,0.238095,734.176677,POINT (6078107.96035099 1988346.45439856)
1,22,0,60329,3,196.000000,El Camino & Palm,12.793264,3.426027,16.219290,POINT (6079055.87151292 1986640.97191715)
2,22,0,60330,4,173.000000,El Camino & Galvez,66.459791,7.428228,73.888019,POINT (6079788.22634742 1985807.8761424)
3,22,0,60331,5,146.500000,El Camino & Sam McDonald,4.579660,0.262500,4.842160,POINT (6080518.11694799 1984980.67143197)
4,22,0,60332,6,158.666667,El Camino & Churchill,4.251791,1.055649,5.307440,POINT (6081284.45957541 1984093.84840249)
...,...,...,...,...,...,...,...,...,...,...
590,523,1,60651,13,-466.000000,Stevens Creek & Cabot,52.817891,62.395022,115.212913,POINT (6127904.45885491 1943613.84307656)
591,523,1,60653,14,-488.500000,NaN,NaN,NaN,NaN,NaN
592,523,1,60656,15,-525.000000,Stevens Creek & Wolfe,66.990445,88.557801,155.548246,POINT (6121122.69649982 1943673.64184149)
593,523,1,60660,16,-604.500000,Stevens Creek & De Anza,85.345256,218.676593,304.021849,POINT (6115865.54428449 1943743.24668914)


In [69]:
stations_df.to_csv("../data/processed/temp.csv")